# Train TinyLlama HelpSteer2 Adapters

This notebook trains independent TinyLlama LoRA specialists for helpfulness, correctness, coherence, complexity, and verbosity. Each specialist starts from the same fresh base model. Training loss and evaluation loss can be followed in TensorBoard.

## 1. Clone or update the repository

In [ ]:
%cd /content
import os
import shutil

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"

if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

## 2. Show repository structure

In [ ]:
!pwd
!ls
!ls scripts
!ls src

## 3. Check the GPU

In [ ]:
!nvidia-smi

## 4. Install dependencies

The pinned pandas and NumPy versions remain compatible with the standard Colab environment. TinyLlama uses standard LoRA training without 4-bit quantization.

In [ ]:
!pip install -q -U "pandas==2.2.2" "numpy<2.1" transformers datasets peft accelerate tensorboard

If Colab upgraded core packages, restart the runtime once after this cell, then rerun the repository cell before training.

## 5. Run a single-attribute smoke test

In [ ]:
!python scripts/train_tinyllama_helpsteer2_adapters.py --attributes helpfulness --split "train[:20]" --num_epochs 1 --use_tensorboard

## 6. Check the saved helpfulness adapter

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes helpfulness

## 7. Open TensorBoard

Open the **Scalars** view to inspect `train_loss` and `eval_loss` against `global_step`. New points appear as training writes logs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/tensorboard/tinyllama_helpsteer2

## 8. Start one adapter training run

The process starts in the background so the stop-control cells remain usable. Train one adapter per command: start `helpfulness`, `correctness`, `coherence`, `complexity`, and `verbosity` manually by changing `--attributes`. This cell starts helpfulness. Progress is written to `/content/tinyllama_helpsteer2_training.log`.

In [ ]:
!rm -f STOP_TRAINING STOP_CURRENT_ADAPTER
!nohup python scripts/train_tinyllama_helpsteer2_adapters.py --attributes helpfulness --split "train[:10000]" --eval_split "train[10000:11000]" --max_training_examples 8434 --num_epochs 5 --batch_size 8 --max_length 2048 --learning_rate 1e-4 --logging_steps 10 --eval_steps 100 --save_steps 500 --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &
!sleep 3
!tail -n 30 /content/tinyllama_helpsteer2_training.log

Run the next cell whenever you want to inspect recent progress.

In [ ]:
!tail -n 50 /content/tinyllama_helpsteer2_training.log

## 9. Stop controls

`STOP_CURRENT_ADAPTER` continues only to the next `save_steps` boundary, saves the current adapter/checkpoint, removes the stop file, and stops this script run. Use `save_steps` to control the maximum wait.

In [ ]:
!touch STOP_CURRENT_ADAPTER

`STOP_TRAINING` also continues only to the next `save_steps` boundary, saves the current adapter/checkpoint, and stops this script run. It is not removed automatically.

In [ ]:
!touch STOP_TRAINING

## 10. Inspect logs and adapters

In [ ]:
!ls results/tinyllama_helpsteer2_training_logs 2>/dev/null || true
!ls results/tensorboard/tinyllama_helpsteer2 2>/dev/null || true
!python scripts/check_tinyllama_helpsteer2_adapters.py

## 11. Git safety

Adapters, checkpoints, TensorBoard events, `.safetensors`, `.bin`, and zip/model files are generated artifacts and must stay out of GitHub.

In [ ]:
!git status --short